# Pertemuan 3 - Data Cleaning dan Akses API

**Nama:** Moch Faris Oldie  
**NIM:** 250401020048  
**Kelas:** IF401

## Tujuan Praktikum

Notebook ini mempraktikkan pembersihan data, penanganan duplikat, normalisasi teks, imputasi missing value, penanganan outlier, ekspor data bersih, dan akses data dari API publik.

In [1]:
import json
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

import numpy as np
import pandas as pd

# A. Membuat dataset kotor
df = pd.DataFrame({
    "kota": [" jakarta ", "JAKARTA", "bandung", "Bandung ", "surabaya", "Surabaya", "medan", "medan", "jakarta "],
    "kondisi": [" Bagus", "bagus", "cukup ", "CUKUP", "rusak", "Rusak ", "bagus", "bagus", " BAGUS"],
    "luasm2": [45, 45, 70, np.nan, 120, 5000, 80, 80, 60],
    "hargajuta": [350, 350, 650, 720, np.nan, 90000, 500, 500, 480],
    "kamar": [2, 2, 3, np.nan, 4, 8, 3, 3, 2],
    "tahunbangun": [2012, 2012, 2018, 2016, np.nan, 1900, 2020, 2020, 2019],
})

print("Shape awal:", df.shape)
print("Missing value awal:")
print(df.isnull().sum())
print("Data awal:")
print(df)

# B. Hapus duplikat
df = df.drop_duplicates().copy()
print()
print("Setelah hapus duplikat:", df.shape)

# C. Normalisasi string
df["kota"] = df["kota"].astype("string").str.strip().str.title()
df["kondisi"] = df["kondisi"].astype("string").str.strip().str.lower()

# D. Imputasi missing values
numeric_cols = ["luasm2", "hargajuta", "kamar", "tahunbangun"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
for col in ["luasm2", "hargajuta", "tahunbangun"]:
    df[col] = df[col].fillna(df[col].median())
mode_kamar = df["kamar"].mode(dropna=True)
df["kamar"] = df["kamar"].fillna(mode_kamar.iloc[0] if not mode_kamar.empty else 2)

# E. Tangani outlier dengan IQR fence
for col in ["hargajuta", "luasm2", "tahunbangun"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df[col] = df[col].clip(lower, upper)

# F. Validasi akhir
print()
print("Missing values total:", df.isnull().sum().sum())
print("Duplikat total:", df.duplicated().sum())
print("Data bersih:")
print(df)

# G. Ekspor data bersih
df.to_csv("housingclean.csv", index=False)
print()
print("Dataset bersih disimpan ke housingclean.csv")

# H. Akses JSONPlaceholder API
url = "https://jsonplaceholder.typicode.com/users"
try:
    with urlopen(url, timeout=10) as response:
        data_users = json.loads(response.read().decode("utf-8"))
    users_df = pd.json_normalize(data_users)
    print()
    print("Data API users:")
    print(users_df.head())
except (HTTPError, URLError, TimeoutError, OSError) as error:
    print()
    print("API tidak dapat diakses saat ini:", error)
    users_df = pd.DataFrame({"id": [1, 2], "name": ["Leanne Graham", "Ervin Howell"], "email": ["leanne@example.com", "ervin@example.com"]})
    print("Memakai contoh data fallback:")
    print(users_df)


Shape awal: (9, 6)
Missing value awal:
kota           0
kondisi        0
luasm2         1
hargajuta      1
kamar          1
tahunbangun    1
dtype: int64
Data awal:
        kota kondisi  luasm2  hargajuta  kamar  tahunbangun
0   jakarta    Bagus    45.0      350.0    2.0       2012.0
1    JAKARTA   bagus    45.0      350.0    2.0       2012.0
2    bandung  cukup     70.0      650.0    3.0       2018.0
3   Bandung    CUKUP     NaN      720.0    NaN       2016.0
4   surabaya   rusak   120.0        NaN    4.0          NaN
5   Surabaya  Rusak   5000.0    90000.0    8.0       1900.0
6      medan   bagus    80.0      500.0    3.0       2020.0
7      medan   bagus    80.0      500.0    3.0       2020.0
8   jakarta    BAGUS    60.0      480.0    2.0       2019.0

Setelah hapus duplikat: (8, 6)

Missing values total: 0
Duplikat total: 1
Data bersih:
       kota kondisi   luasm2  hargajuta  kamar  tahunbangun
0   Jakarta   bagus   45.000      350.0    2.0     2012.000
1   Jakarta   bagus   45.00

## Kesimpulan

Pada pertemuan ini saya mempelajari tahapan data cleaning, mulai dari menghapus duplikat, menyeragamkan format teks, mengisi missing value, dan membatasi outlier dengan metode IQR. Saya juga belajar mengambil data dari API publik dan mengubah JSON menjadi DataFrame. Keterbatasannya adalah dataset rumah dibuat sintetis, sehingga hasilnya belum mewakili kondisi pasar properti sebenarnya.